# Adaptive Lower Controller

Three focused adaptive DQN experiments: rear-flow pressure, traffic-conformity reward, and a safe-speed controller formulation.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
helper_dir_str = str(HELPER_DIR)
if helper_dir_str not in sys.path:
    sys.path.insert(0, helper_dir_str)

from dqn_notebook_utils import (
    build_dqn_args,
    build_env_config,
    evaluate_saved_model,
    load_dqn_backend,
    show_policy_panel,
    train_and_display,
)

trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
    backend_module="elurant_dqn",
    notebook_subdir="adaptive_lower_controller",
    results_subdir="adaptive_lower_controller",
)

## Common Config

In [ ]:
import os
import sys
from pathlib import Path

if "build_env_config" not in globals() or "RESULTS_DIR" not in globals():
    def find_project_root() -> Path:
        for candidate in [Path.cwd(), *Path.cwd().parents]:
            if (candidate / "src").exists() and (candidate / ".git").exists():
                return candidate
        raise RuntimeError("Could not locate the project root.")

    PROJECT_ROOT = find_project_root()
    HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
    helper_dir_str = str(HELPER_DIR)
    if helper_dir_str not in sys.path:
        sys.path.insert(0, helper_dir_str)

    from dqn_notebook_utils import (
        build_dqn_args,
        build_env_config,
        evaluate_saved_model,
        load_dqn_backend,
        train_and_display,
    )

    trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
        backend_module="elurant_dqn",
        notebook_subdir="adaptive_lower_controller",
        results_subdir="adaptive_lower_controller",
    )

environment_profile = "structured_baseline"
eval_environment_profile = environment_profile

environment_overrides = {
    "lanes_count": 3,
    "vehicles_count": 20,
    "duration": 40,
    "ego_spacing": 2.0,
    "vehicles_density": 1.0,
    "simulation_frequency": 15,
    "policy_frequency": 1,
    "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
    "initial_lane_id": None,
    "offroad_terminal": False,
}
# Evaluation environment: choose a profile independently from training.
# Leave overrides empty unless you intentionally want to override that
# eval profile's defaults.
eval_environment_overrides = {}

observation_config = {
    "vehicles_count": 5,
    "features": ["presence", "x", "y", "vx", "vy"],
    "absolute": False,
}
action_config = {
    "type": "DiscreteMetaAction",
}
reward_config = {
    "collision_reward": -1.0,
    "right_lane_reward": 0.1,
    "high_speed_reward": 0.4,
    "lane_change_reward": 0.0,
    "normalize_reward": True,
}
min_reward_speed = 20.0
max_reward_speed = 30.0
speed_config = {
    "reward_speed_range": [min_reward_speed, max_reward_speed],
}

timesteps = 20_000
n_envs = min(4, os.cpu_count() or 1)
eval_episodes = 5
seed = 42
train_freq = 4
gradient_steps = train_freq * n_envs

base_adaptive_longitudinal_config = {
    "enabled": True,
    "mode": "delta",
    "ttc_midpoint": 4.0,
    "ttc_temperature": 1.0,
    "ttc_cap": 10.0,
    "min_target_speed": 10.0,
    "max_target_speed": 35.0,
    "faster_max_delta": 1.25,
    "slower_min_delta": 1.25,
    "slower_max_delta": 2.5,
}
rear_flow_config = {
    "enabled": True,
    "spawn_on_reset": True,
    "spawn_during_episode": True,
    "vehicles_per_lane": 1,
    "lanes": "ego_and_adjacent",
    "distance_range": [25.0, 70.0],
    "speed_offset_range": [2.0, 6.0],
    "absolute_speed_range": [23.0, 34.0],
    "min_lane_gap": 18.0,
    "spawn_probability": 0.35,
    "cooldown_policy_steps": 3,
    "min_ego_progress": 80.0,
    "max_extra_vehicles": 12,
}
traffic_flow_reward_config = {
    "enabled": True,
    "penalty_weight": 0.12,
    "speed_tolerance": 2.0,
    "max_penalty": 0.8,
    "front_ttc_safe": 4.0,
    "rear_ttc_pressure": 5.0,
    "ttc_cap": 10.0,
    "rear_pressure_floor": 0.25,
    "flow_radius": 120.0,
    "lanes": "ego_and_adjacent",
}
hyperparameters = {
    "learning_rate": 2.5e-4,
    "buffer_size": 50_000,
    "learning_starts": 2_000,
    "batch_size": 64,
    "gamma": 0.95,
    "target_update_interval": 1_000,
    "train_freq": train_freq,
    "gradient_steps": gradient_steps,
    "exploration_fraction": 0.70,
    "exploration_final_eps": 0.10,
    "progress_every": 1_000,
    "verbose": 0,
}

saved_model_eval_episodes = 1000
render_trial_episodes = 5
render_trial_seed = seed + 30_000
render_trial_env_config = build_env_config(
    profile_name=environment_profile,
    profile_overrides=environment_overrides,
    observation=observation_config,
    action=action_config,
    reward=reward_config,
    speed=speed_config,
    adaptive_longitudinal={"enabled": False},
    rear_flow=rear_flow_config,
    traffic_flow_reward={"enabled": False},
)

def configure_adaptive_experiment(
    *,
    run_name: str,
    adaptive_overrides: dict | None = None,
    rear_flow: dict | None = None,
    traffic_flow_reward: dict | None = None,
    ttc_observation: dict | None = None,
    seed_offset: int = 0,
) -> dict:
    adaptive_config = {
        **base_adaptive_longitudinal_config,
        **dict(adaptive_overrides or {}),
    }
    training_env_config = build_env_config(
        profile_name=environment_profile,
        profile_overrides=environment_overrides,
        observation=observation_config,
        action=action_config,
        reward=reward_config,
        speed=speed_config,
        adaptive_longitudinal=adaptive_config,
        rear_flow=rear_flow,
        traffic_flow_reward=traffic_flow_reward,
        ttc_observation=ttc_observation,
    )
    saved_model_eval_env_config = build_env_config(
        profile_name=eval_environment_profile,
        profile_overrides=eval_environment_overrides,
        observation=observation_config,
        action=action_config,
        reward=reward_config,
        speed=speed_config,
        adaptive_longitudinal=adaptive_config,
        rear_flow=rear_flow,
        traffic_flow_reward=traffic_flow_reward,
        ttc_observation=ttc_observation,
    )
    args = build_dqn_args(
        results_dir=RESULTS_DIR,
        run_name=run_name,
        timesteps=timesteps,
        eval_episodes=eval_episodes,
        seed=seed + seed_offset,
        num_envs=n_envs,
        device=DEFAULT_DEVICE,
        hyperparameters=hyperparameters,
    )
    return {
        "run_name": run_name,
        "args": args,
        "training_env_config": training_env_config,
        "saved_model_eval_env_config": saved_model_eval_env_config,
        "saved_model_eval_seed": seed + 10000 + seed_offset,
        "saved_model_eval_name": f"saved_model_eval_{saved_model_eval_episodes}_episodes",
    }

def run_adaptive_experiment(experiment: dict, label: str) -> dict:
    display(pd.DataFrame({
        "training": pd.Series(experiment["training_env_config"]),
        "saved_eval": pd.Series(experiment["saved_model_eval_env_config"]),
    }))
    summary = train_and_display(
        trainer,
        experiment["args"],
        experiment["training_env_config"],
        label=label,
    )
    saved_eval_summary = evaluate_saved_model(
        trainer,
        summary_path=RESULTS_DIR / experiment["run_name"] / "summary.json",
        env_config=experiment["saved_model_eval_env_config"],
        episodes=saved_model_eval_episodes,
        seed=experiment["saved_model_eval_seed"],
        name=experiment["saved_model_eval_name"],
        label=label,
    )
    return {"summary": summary, "saved_eval_summary": saved_eval_summary}


### Rear-Aware Observation Override

In [ ]:
# Local to this notebook: widen the kinematic observation and include rear vehicles.
observation_config = {
    "vehicles_count": 10,
    "features": ["presence", "x", "y", "vx", "vy"],
    "absolute": False,
    "see_behind": True,
}

render_trial_env_config = build_env_config(
    profile_name=environment_profile,
    profile_overrides=environment_overrides,
    observation=observation_config,
    action=action_config,
    reward=reward_config,
    speed=speed_config,
    adaptive_longitudinal={"enabled": False},
    rear_flow=rear_flow_config,
    traffic_flow_reward={"enabled": False},
)

display(pd.Series(observation_config, name="rear_aware_observation"))


## RENDER TRIAL

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd


from highway_env.vehicle.behavior import IDMVehicle


if "build_env_config" not in globals() or "RESULTS_DIR" not in globals():
    def find_project_root() -> Path:
        for candidate in [Path.cwd(), *Path.cwd().parents]:
            if (candidate / "src").exists() and (candidate / ".git").exists():
                return candidate
        raise RuntimeError("Could not locate the project root.")

    PROJECT_ROOT = find_project_root()
    HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
    helper_dir_str = str(HELPER_DIR)
    if helper_dir_str not in sys.path:
        sys.path.insert(0, helper_dir_str)

    from dqn_notebook_utils import build_env_config, load_dqn_backend

    trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
        backend_module="elurant_dqn",
        notebook_subdir="adaptive_lower_controller",
        results_subdir="adaptive_lower_controller",
    )


def handoff_ego_to_idm(env) -> None:
    ego_vehicle = env.unwrapped.vehicle
    if ego_vehicle is None:
        raise RuntimeError("No ego vehicle found to convert into IDM behavior.")
    npc_ego = IDMVehicle.create_from(ego_vehicle)
    npc_ego.randomize_behavior()

    road_vehicles = env.unwrapped.road.vehicles
    for idx, vehicle in enumerate(road_vehicles):
        if vehicle is ego_vehicle:
            road_vehicles[idx] = npc_ego
            break
    env.unwrapped.controlled_vehicles = [npc_ego]
    env.unwrapped.vehicle = npc_ego
    env.unwrapped.action_type.controlled_vehicle = npc_ego
    env.unwrapped.observation_type.observer_vehicle = npc_ego


if "render_trial_env_config" not in globals():
    render_trial_episodes = 5
    render_trial_seed = 42 + 30_000
    render_trial_env_config = build_env_config(
        profile_name="structured_baseline",
        profile_overrides={
            "lanes_count": 3,
            "vehicles_count": 20,
            "duration": 40,
            "ego_spacing": 2.0,
            "vehicles_density": 1.0,
            "simulation_frequency": 15,
            "policy_frequency": 1,
            "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
            "initial_lane_id": None,
            "offroad_terminal": False,
        },
        observation={
            "vehicles_count": 10,
            "features": ["presence", "x", "y", "vx", "vy"],
            "absolute": False,
            "see_behind": True,
        },
        action={"type": "DiscreteMetaAction"},
        reward={
            "collision_reward": -1.0,
            "right_lane_reward": 0.1,
            "high_speed_reward": 0.4,
            "lane_change_reward": 0.0,
            "normalize_reward": True,
        },
        speed={"reward_speed_range": [20.0, 30.0]},
        adaptive_longitudinal={"enabled": False},
        rear_flow={
            "enabled": True,
            "spawn_on_reset": True,
            "spawn_during_episode": True,
            "vehicles_per_lane": 1,
            "lanes": "ego_and_adjacent",
            "distance_range": [25.0, 70.0],
            "speed_offset_range": [2.0, 6.0],
            "absolute_speed_range": [23.0, 34.0],
            "min_lane_gap": 18.0,
            "spawn_probability": 0.35,
            "cooldown_policy_steps": 3,
            "min_ego_progress": 80.0,
            "max_extra_vehicles": 12,
        },
        traffic_flow_reward={"enabled": False},
    )

print("Render trial env config:")
print(json.dumps(render_trial_env_config, indent=2))

render_env = trainer.make_env(render_mode="human", config=render_trial_env_config)
idle_action = render_env.unwrapped.action_type.actions_indexes["IDLE"]
render_trial_rows = []

try:
    for episode in range(render_trial_episodes):
        obs, info = render_env.reset(seed=render_trial_seed + episode)
        handoff_ego_to_idm(render_env)
        render_env.render()

        done = False
        truncated = False
        step_count = 0
        rear_spawn_total = int(info.get("rear_flow_spawned_count", 0))
        final_info = dict(info)

        while not (done or truncated):
            obs, reward, done, truncated, info = render_env.step(idle_action)
            render_env.render()
            step_count += 1
            rear_spawn_total += int(info.get("rear_flow_spawned_count", 0))
            final_info = dict(info)

        row = {
            "episode": episode + 1,
            "steps": step_count,
            "collision": bool(final_info.get("crashed", getattr(render_env.unwrapped.vehicle, "crashed", False))),
            "rear_flow_spawned_total": rear_spawn_total,
            "final_speed": float(getattr(render_env.unwrapped.vehicle, "speed", 0.0)),
        }
        render_trial_rows.append(row)
        print(row)
finally:
    render_env.close()

display(pd.DataFrame(render_trial_rows))

In [ ]:
# Evaluation-only experiment: trained baseline DQN with and without the adaptive lower-level controller.
import json
import sys
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from stable_baselines3 import DQN


if "PROJECT_ROOT" not in globals():
    def find_project_root() -> Path:
        for candidate in [Path.cwd(), *Path.cwd().parents]:
            if (candidate / "src").exists() and (candidate / ".git").exists():
                return candidate
        raise RuntimeError("Could not locate the project root.")

    PROJECT_ROOT = find_project_root()

HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
helper_dir_str = str(HELPER_DIR)
if helper_dir_str not in sys.path:
    sys.path.insert(0, helper_dir_str)

from dqn_notebook_utils import build_metric_summary, load_dqn_backend

if "trainer" not in globals() or "RESULTS_DIR" not in globals():
    trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
        backend_module="elurant_dqn",
        notebook_subdir="adaptive_lower_controller",
        results_subdir="adaptive_lower_controller",
    )

from adaptive_longitudinal import compute_same_lane_ttc


baseline_summary_path = (
    PROJECT_ROOT
    / "artifacts"
    / "dqn"
    / "baseline_dqn"
    / "baseline_dqn_tuned_20k"
    / "summary.json"
)
if not baseline_summary_path.exists():
    raise FileNotFoundError(f"Baseline DQN summary not found: {baseline_summary_path}")

baseline_summary = json.loads(baseline_summary_path.read_text(encoding="utf-8"))
baseline_model_path = Path(baseline_summary["model_path"])
if not baseline_model_path.exists():
    raise FileNotFoundError(f"Baseline DQN model not found: {baseline_model_path}")

baseline_model = DQN.load(str(baseline_model_path), device=DEFAULT_DEVICE)
baseline_eval_env_config = deepcopy(baseline_summary["env_config"])

fallback_adaptive_config = {
    "enabled": True,
    "mode": "delta",
    "ttc_midpoint": 4.0,
    "ttc_temperature": 1.0,
    "ttc_cap": 10.0,
    "min_target_speed": 10.0,
    "max_target_speed": 35.0,
    "faster_max_delta": 1.25,
    "slower_min_delta": 1.25,
    "slower_max_delta": 2.5,
    "cruise_speed": 28.0,
    "action_speed_delta": 3.0,
}
adaptive_controller_config = {
    **fallback_adaptive_config,
    **dict(globals().get("base_adaptive_longitudinal_config", {})),
    "enabled": True,
    "mode": "delta",
}

lower_controller_eval_env_config = deepcopy(baseline_eval_env_config)
lower_controller_eval_env_config["adaptive_longitudinal"] = adaptive_controller_config
lower_controller_eval_env_config["rear_flow"] = {"enabled": False}
lower_controller_eval_env_config["traffic_flow_reward"] = {"enabled": False}

baseline_controller_eval_episodes = 1000
baseline_controller_eval_seed = int(baseline_summary.get("seed", 42)) + 40_000
baseline_controller_eval_dir = RESULTS_DIR / "baseline_dqn_with_lower_controller_eval_1000"
baseline_controller_eval_dir.mkdir(parents=True, exist_ok=True)


def action_index(action) -> int:
    return int(np.asarray(action).item())


def action_label(env, index: int) -> str:
    actions = getattr(env.unwrapped.action_type, "actions", {})
    if isinstance(actions, dict):
        return str(actions.get(int(index), index))
    if 0 <= int(index) < len(actions):
        return str(actions[int(index)])
    return str(index)


def update_overtake_tracker(env, seen_ahead_ids: set[int], overtaken_ids: set[int]) -> None:
    ego_vehicle = getattr(env.unwrapped, "vehicle", None)
    road = getattr(env.unwrapped, "road", None)
    if ego_vehicle is None or road is None:
        return
    ego_x = float(ego_vehicle.position[0])
    for other in road.vehicles:
        if other is ego_vehicle:
            continue
        other_id = id(other)
        dx = float(other.position[0] - ego_x)
        if dx > 0.0:
            seen_ahead_ids.add(other_id)
        elif other_id in seen_ahead_ids:
            overtaken_ids.add(other_id)


def evaluate_policy_with_trace(
    *,
    label: str,
    model: DQN,
    env_config: dict,
    episodes: int,
    seed: int,
    ttc_cap: float = 10.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    env = trainer.make_env(render_mode=None, config=env_config)
    policy_frequency = float(env_config.get("policy_frequency", 1.0)) or 1.0
    summary_rows = []
    trace_rows = []

    try:
        for episode_idx in range(int(episodes)):
            obs, _ = env.reset(seed=int(seed) + episode_idx)
            terminated = False
            truncated = False
            step = 0
            total_reward = 0.0
            speed_trace = []
            ttc_trace = []
            adaptive_delta_trace = []
            adaptive_target_trace = []
            adaptive_ttc_trace = []
            adaptive_action_steps = 0
            seen_ahead_ids: set[int] = set()
            overtaken_ids: set[int] = set()
            final_info = {}

            while not (terminated or truncated):
                update_overtake_tracker(env, seen_ahead_ids, overtaken_ids)
                ego_vehicle = env.unwrapped.vehicle
                speed = float(getattr(ego_vehicle, "speed", 0.0))
                target_speed = float(getattr(ego_vehicle, "target_speed", speed))
                ttc = compute_same_lane_ttc(env, ttc_cap=ttc_cap)
                predicted_action, _ = model.predict(obs, deterministic=True)
                action_id = action_index(predicted_action)
                requested_action = action_label(env, action_id)

                obs, reward, terminated, truncated, info = env.step(predicted_action)
                total_reward += float(reward)
                final_info = dict(info)
                forwarded_action = str(final_info.get("adaptive_forwarded_action", requested_action))
                adaptive_delta = float(final_info.get("adaptive_speed_delta", np.nan))
                target_after = float(final_info.get("adaptive_target_speed_after", np.nan))
                controller_ttc = float(final_info.get("adaptive_ttc", np.nan))

                speed_trace.append(speed)
                ttc_trace.append(ttc)
                if "adaptive_speed_delta" in final_info:
                    adaptive_delta_trace.append(adaptive_delta)
                    adaptive_target_trace.append(target_after)
                    adaptive_ttc_trace.append(controller_ttc)
                    if final_info.get("adaptive_requested_action") in {"FASTER", "SLOWER"}:
                        adaptive_action_steps += 1

                trace_rows.append(
                    {
                        "variant": label,
                        "episode": episode_idx + 1,
                        "step": step,
                        "time_s": step / policy_frequency,
                        "speed": speed,
                        "ttc": ttc,
                        "target_speed_before": target_speed,
                        "target_speed_after": target_after,
                        "adaptive_speed_delta": adaptive_delta,
                        "requested_action": requested_action,
                        "forwarded_action": forwarded_action,
                        "reward": float(reward),
                    }
                )
                step += 1

            collision = bool(final_info.get("crashed", getattr(env.unwrapped.vehicle, "crashed", False)))
            row = {
                "variant": label,
                "episode": episode_idx + 1,
                "reward": float(total_reward),
                "collision": collision,
                "avg_speed": float(np.mean(speed_trace)) if speed_trace else 0.0,
                "overtakes": int(len(overtaken_ids)),
                "avg_ttc": float(np.mean(ttc_trace)) if ttc_trace else float(ttc_cap),
                "min_ttc": float(np.min(ttc_trace)) if ttc_trace else float(ttc_cap),
                "steps": int(step),
            }
            if adaptive_delta_trace:
                row.update(
                    {
                        "adaptive_longitudinal_steps": int(adaptive_action_steps),
                        "adaptive_avg_speed_delta": float(np.nanmean(adaptive_delta_trace)),
                        "adaptive_avg_target_speed": float(np.nanmean(adaptive_target_trace)),
                        "adaptive_avg_controller_ttc": float(np.nanmean(adaptive_ttc_trace)),
                    }
                )
            summary_rows.append(row)

            if (episode_idx + 1) % 100 == 0:
                print(f"{label}: evaluated {episode_idx + 1}/{episodes} episodes")
    finally:
        env.close()

    return pd.DataFrame(summary_rows), pd.DataFrame(trace_rows)


def plot_performance_comparison(summary_df: pd.DataFrame, save_path: Path) -> None:
    specs = [
        ("reward", "Reward", 1.0),
        ("collision", "Collision rate (%)", 100.0),
        ("avg_speed", "Avg speed (m/s)", 1.0),
        ("avg_ttc", "Avg TTC (s)", 1.0),
        ("min_ttc", "Min TTC (s)", 1.0),
        ("overtakes", "Overtakes", 1.0),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    for ax, (column, title, scale) in zip(axes.flat, specs):
        grouped = summary_df.groupby("variant")[column].mean().astype(float) * scale
        ax.bar(grouped.index, grouped.values, color=["tab:gray", "tab:blue"], alpha=0.85)
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=15)
        ax.grid(axis="y", alpha=0.3)
    fig.suptitle("Baseline trained DQN: raw policy vs adaptive lower-level controller")
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_velocity_ttc(trace_df: pd.DataFrame, *, variant: str, save_path: Path, episode: int | None = None) -> None:
    subset = trace_df[trace_df["variant"] == variant].copy()
    if episode is None:
        plot_df = subset.groupby("time_s", as_index=False).agg(
            speed=("speed", "mean"),
            ttc=("ttc", "mean"),
            target_speed_after=("target_speed_after", "mean"),
        )
        title_suffix = "mean over 1,000 episodes"
    else:
        plot_df = subset[subset["episode"] == int(episode)].copy()
        title_suffix = f"episode {episode}"

    fig, ax_speed = plt.subplots(figsize=(11, 4.8))
    ax_ttc = ax_speed.twinx()
    ax_speed.plot(plot_df["time_s"], plot_df["speed"], color="tab:blue", label="ego speed")
    if "target_speed_after" in plot_df and plot_df["target_speed_after"].notna().any():
        ax_speed.plot(
            plot_df["time_s"],
            plot_df["target_speed_after"],
            color="tab:cyan",
            linestyle="--",
            label="target speed",
        )
    ax_ttc.plot(plot_df["time_s"], plot_df["ttc"], color="tab:red", label="front TTC")
    ax_speed.set_xlabel("Time (s)")
    ax_speed.set_ylabel("Velocity (m/s)")
    ax_ttc.set_ylabel("TTC (s)")
    ax_speed.set_title(f"{variant}: velocity and TTC over time ({title_suffix})")
    lines_1, labels_1 = ax_speed.get_legend_handles_labels()
    lines_2, labels_2 = ax_ttc.get_legend_handles_labels()
    ax_speed.legend(lines_1 + lines_2, labels_1 + labels_2, loc="best")
    ax_speed.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


summary_parts = []
trace_parts = []
for variant_label, variant_config in [
    ("Baseline DQN", baseline_eval_env_config),
    ("Baseline DQN + lower controller", lower_controller_eval_env_config),
]:
    episode_df, trace_df = evaluate_policy_with_trace(
        label=variant_label,
        model=baseline_model,
        env_config=variant_config,
        episodes=baseline_controller_eval_episodes,
        seed=baseline_controller_eval_seed,
        ttc_cap=float(variant_config.get("adaptive_longitudinal", {}).get("ttc_cap", 10.0)),
    )
    safe_name = variant_label.lower().replace(" + ", "_").replace(" ", "_")
    episode_df.to_json(
        baseline_controller_eval_dir / f"{safe_name}_episode_metrics.json",
        orient="records",
        indent=2,
    )
    trace_df.to_json(
        baseline_controller_eval_dir / f"{safe_name}_step_trace.json",
        orient="records",
        indent=2,
    )
    summary_parts.append(episode_df)
    trace_parts.append(trace_df)

baseline_controller_summary_df = pd.concat(summary_parts, ignore_index=True)
baseline_controller_trace_df = pd.concat(trace_parts, ignore_index=True)
baseline_controller_summary_df.to_json(
    baseline_controller_eval_dir / "comparison_episode_metrics.json",
    orient="records",
    indent=2,
)
baseline_controller_trace_df.to_json(
    baseline_controller_eval_dir / "comparison_step_trace.json",
    orient="records",
    indent=2,
)

comparison_stats = (
    baseline_controller_summary_df.groupby("variant")
    .agg(
        episodes=("episode", "count"),
        mean_reward=("reward", "mean"),
        collision_rate_percent=("collision", lambda values: 100.0 * float(np.mean(values.astype(bool)))),
        avg_speed=("avg_speed", "mean"),
        avg_ttc=("avg_ttc", "mean"),
        min_ttc=("min_ttc", "mean"),
        overtakes=("overtakes", "mean"),
        adaptive_longitudinal_steps=("adaptive_longitudinal_steps", "mean"),
        adaptive_avg_speed_delta=("adaptive_avg_speed_delta", "mean"),
        adaptive_avg_target_speed=("adaptive_avg_target_speed", "mean"),
        adaptive_avg_controller_ttc=("adaptive_avg_controller_ttc", "mean"),
    )
    .reset_index()
)
comparison_stats.to_json(baseline_controller_eval_dir / "comparison_summary.json", orient="records", indent=2)

performance_plot_path = baseline_controller_eval_dir / "performance_comparison.png"
plot_performance_comparison(baseline_controller_summary_df, performance_plot_path)

mean_trace_plot_paths = []
worst_trace_plot_paths = []
for variant_label in baseline_controller_summary_df["variant"].unique():
    safe_name = variant_label.lower().replace(" + ", "_").replace(" ", "_")
    mean_path = baseline_controller_eval_dir / f"{safe_name}_mean_velocity_ttc.png"
    plot_velocity_ttc(baseline_controller_trace_df, variant=variant_label, save_path=mean_path)
    mean_trace_plot_paths.append(mean_path)

    variant_summary = baseline_controller_summary_df[baseline_controller_summary_df["variant"] == variant_label]
    worst_episode = int(variant_summary.loc[variant_summary["min_ttc"].idxmin(), "episode"])
    worst_path = baseline_controller_eval_dir / f"{safe_name}_lowest_ttc_episode_velocity_ttc.png"
    plot_velocity_ttc(
        baseline_controller_trace_df,
        variant=variant_label,
        save_path=worst_path,
        episode=worst_episode,
    )
    worst_trace_plot_paths.append(worst_path)

print(f"Baseline model: {baseline_model_path}")
print(f"Saved comparison outputs to: {baseline_controller_eval_dir}")
display(comparison_stats.round(3))
display(Image(filename=performance_plot_path))
for plot_path in mean_trace_plot_paths + worst_trace_plot_paths:
    display(Image(filename=plot_path))

display(
    baseline_controller_summary_df[
        [
            "variant",
            "episode",
            "reward",
            "collision",
            "avg_speed",
            "avg_ttc",
            "min_ttc",
            "overtakes",
            "adaptive_longitudinal_steps",
            "adaptive_avg_speed_delta",
            "adaptive_avg_target_speed",
        ]
    ]
    .head(10)
    .round(3)
)


## Experiment 1: Rear Flow Environment

In [ ]:
if "configure_adaptive_experiment" not in globals():
    import os
    import sys
    from pathlib import Path

    import pandas as pd

    if "build_env_config" not in globals() or "RESULTS_DIR" not in globals():
        def find_project_root() -> Path:
            for candidate in [Path.cwd(), *Path.cwd().parents]:
                if (candidate / "src").exists() and (candidate / ".git").exists():
                    return candidate
            raise RuntimeError("Could not locate the project root.")

        PROJECT_ROOT = find_project_root()
        HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
        helper_dir_str = str(HELPER_DIR)
        if helper_dir_str not in sys.path:
            sys.path.insert(0, helper_dir_str)

        from dqn_notebook_utils import (
            build_dqn_args,
            build_env_config,
            evaluate_saved_model,
            load_dqn_backend,
            train_and_display,
        )

        trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
            backend_module="elurant_dqn",
            notebook_subdir="adaptive_lower_controller",
            results_subdir="adaptive_lower_controller",
        )

    environment_profile = "structured_baseline"
    eval_environment_profile = environment_profile

    environment_overrides = {
        "lanes_count": 3,
        "vehicles_count": 20,
        "duration": 40,
        "ego_spacing": 2.0,
        "vehicles_density": 1.0,
        "simulation_frequency": 15,
        "policy_frequency": 1,
        "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
        "initial_lane_id": None,
        "offroad_terminal": False,
    }
    # Evaluation environment: choose a profile independently from training.
    # Leave overrides empty unless you intentionally want to override that
    # eval profile's defaults.
    eval_environment_overrides = {}

    # Rear-Aware Observation Override (local to this notebook)
    observation_config = {
        "vehicles_count": 10,
        "features": ["presence", "x", "y", "vx", "vy"],
        "absolute": False,
        "see_behind": True,
    }
    action_config = {
        "type": "DiscreteMetaAction",
    }
    reward_config = {
        "collision_reward": -1.0,
        "right_lane_reward": 0.1,
        "high_speed_reward": 0.4,
        "lane_change_reward": 0.0,
        "normalize_reward": True,
    }
    min_reward_speed = 20.0
    max_reward_speed = 30.0
    speed_config = {
        "reward_speed_range": [min_reward_speed, max_reward_speed],
    }

    timesteps = 20_000
    n_envs = min(4, os.cpu_count() or 1)
    eval_episodes = 5
    seed = 42
    train_freq = 4
    gradient_steps = train_freq * n_envs

    base_adaptive_longitudinal_config = {
        "enabled": True,
        "mode": "delta",
        "ttc_midpoint": 4.0,
        "ttc_temperature": 1.0,
        "ttc_cap": 10.0,
        "min_target_speed": 10.0,
        "max_target_speed": 35.0,
        "faster_max_delta": 1.25,
        "slower_min_delta": 1.25,
        "slower_max_delta": 2.5,
    }
    rear_flow_config = {
        "enabled": True,
        "spawn_on_reset": True,
        "spawn_during_episode": True,
        "vehicles_per_lane": 1,
        "lanes": "ego_and_adjacent",
        "distance_range": [25.0, 70.0],
        "speed_offset_range": [2.0, 6.0],
        "absolute_speed_range": [23.0, 34.0],
        "min_lane_gap": 18.0,
        "spawn_probability": 0.35,
        "cooldown_policy_steps": 3,
        "min_ego_progress": 80.0,
        "max_extra_vehicles": 12,
    }
    traffic_flow_reward_config = {
        "enabled": True,
        "penalty_weight": 0.12,
        "speed_tolerance": 2.0,
        "max_penalty": 0.8,
        "front_ttc_safe": 4.0,
        "rear_ttc_pressure": 5.0,
        "ttc_cap": 10.0,
        "rear_pressure_floor": 0.25,
        "flow_radius": 120.0,
        "lanes": "ego_and_adjacent",
    }
    hyperparameters = {
        "learning_rate": 2.5e-4,
        "buffer_size": 50_000,
        "learning_starts": 2_000,
        "batch_size": 64,
        "gamma": 0.95,
        "target_update_interval": 1_000,
        "train_freq": train_freq,
        "gradient_steps": gradient_steps,
        "exploration_fraction": 0.70,
        "exploration_final_eps": 0.10,
        "progress_every": 1_000,
        "verbose": 0,
    }

    saved_model_eval_episodes = 1000

    def configure_adaptive_experiment(
        *,
        run_name: str,
        adaptive_overrides: dict | None = None,
        rear_flow: dict | None = None,
        traffic_flow_reward: dict | None = None,
        ttc_observation: dict | None = None,
        seed_offset: int = 0,
    ) -> dict:
        adaptive_config = {
            **base_adaptive_longitudinal_config,
            **dict(adaptive_overrides or {}),
        }
        training_env_config = build_env_config(
            profile_name=environment_profile,
            profile_overrides=environment_overrides,
            observation=observation_config,
            action=action_config,
            reward=reward_config,
            speed=speed_config,
            adaptive_longitudinal=adaptive_config,
            rear_flow=rear_flow,
            traffic_flow_reward=traffic_flow_reward,
            ttc_observation=ttc_observation,
        )
        saved_model_eval_env_config = build_env_config(
            profile_name=eval_environment_profile,
            profile_overrides=eval_environment_overrides,
            observation=observation_config,
            action=action_config,
            reward=reward_config,
            speed=speed_config,
            adaptive_longitudinal=adaptive_config,
            rear_flow=rear_flow,
            traffic_flow_reward=traffic_flow_reward,
            ttc_observation=ttc_observation,
        )
        args = build_dqn_args(
            results_dir=RESULTS_DIR,
            run_name=run_name,
            timesteps=timesteps,
            eval_episodes=eval_episodes,
            seed=seed + seed_offset,
            num_envs=n_envs,
            device=DEFAULT_DEVICE,
            hyperparameters=hyperparameters,
        )
        return {
            "run_name": run_name,
            "args": args,
            "training_env_config": training_env_config,
            "saved_model_eval_env_config": saved_model_eval_env_config,
            "saved_model_eval_seed": seed + 10000 + seed_offset,
            "saved_model_eval_name": f"saved_model_eval_{saved_model_eval_episodes}_episodes",
        }

    def run_adaptive_experiment(experiment: dict, label: str) -> dict:
        display(pd.DataFrame({
            "training": pd.Series(experiment["training_env_config"]),
            "saved_eval": pd.Series(experiment["saved_model_eval_env_config"]),
        }))
        summary = train_and_display(
            trainer,
            experiment["args"],
            experiment["training_env_config"],
            label=label,
        )
        saved_eval_summary = evaluate_saved_model(
            trainer,
            summary_path=RESULTS_DIR / experiment["run_name"] / "summary.json",
            env_config=experiment["saved_model_eval_env_config"],
            episodes=saved_model_eval_episodes,
            seed=experiment["saved_model_eval_seed"],
            name=experiment["saved_model_eval_name"],
            label=label,
        )
        return {"summary": summary, "saved_eval_summary": saved_eval_summary}

rear_flow_only = configure_adaptive_experiment(
run_name="adaptive_rear_flow_env_20k",
adaptive_overrides={"mode": "delta"},
rear_flow=rear_flow_config,
traffic_flow_reward={"enabled": False},
seed_offset=0,
)
rear_flow_only_results = run_adaptive_experiment(
rear_flow_only,
label="Adaptive DQN - Rear Flow Env",
)


## Experiment 2: Rear Flow + Traffic Reward

In [ ]:
if "configure_adaptive_experiment" not in globals():
    import os
    import sys
    from pathlib import Path

    import pandas as pd

    if "build_env_config" not in globals() or "RESULTS_DIR" not in globals():
        def find_project_root() -> Path:
            for candidate in [Path.cwd(), *Path.cwd().parents]:
                if (candidate / "src").exists() and (candidate / ".git").exists():
                    return candidate
            raise RuntimeError("Could not locate the project root.")

        PROJECT_ROOT = find_project_root()
        HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
        helper_dir_str = str(HELPER_DIR)
        if helper_dir_str not in sys.path:
            sys.path.insert(0, helper_dir_str)

        from dqn_notebook_utils import (
            build_dqn_args,
            build_env_config,
            evaluate_saved_model,
            load_dqn_backend,
            train_and_display,
        )

        trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
            backend_module="elurant_dqn",
            notebook_subdir="adaptive_lower_controller",
            results_subdir="adaptive_lower_controller",
        )

    environment_profile = "structured_baseline"
    eval_environment_profile = environment_profile

    environment_overrides = {
        "lanes_count": 3,
        "vehicles_count": 20,
        "duration": 40,
        "ego_spacing": 2.0,
        "vehicles_density": 1.0,
        "simulation_frequency": 15,
        "policy_frequency": 1,
        "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
        "initial_lane_id": None,
        "offroad_terminal": False,
    }
    # Evaluation environment: choose a profile independently from training.
    # Leave overrides empty unless you intentionally want to override that
    # eval profile's defaults.
    eval_environment_overrides = {}

    observation_config = {
        "vehicles_count": 10,
        "features": ["presence", "x", "y", "vx", "vy"],
        "absolute": False,
        "see_behind": True,
    }
    action_config = {
        "type": "DiscreteMetaAction",
    }
    reward_config = {
        "collision_reward": -1.0,
        "right_lane_reward": 0.1,
        "high_speed_reward": 0.4,
        "lane_change_reward": 0.0,
        "normalize_reward": True,
    }
    min_reward_speed = 20.0
    max_reward_speed = 30.0
    speed_config = {
        "reward_speed_range": [min_reward_speed, max_reward_speed],
    }

    timesteps = 20_000
    n_envs = min(4, os.cpu_count() or 1)
    eval_episodes = 5
    seed = 42
    train_freq = 4
    gradient_steps = train_freq * n_envs

    base_adaptive_longitudinal_config = {
        "enabled": True,
        "mode": "delta",
        "ttc_midpoint": 4.0,
        "ttc_temperature": 1.0,
        "ttc_cap": 10.0,
        "min_target_speed": 10.0,
        "max_target_speed": 35.0,
        "faster_max_delta": 1.25,
        "slower_min_delta": 1.25,
        "slower_max_delta": 2.5,
    }
    rear_flow_config = {
        "enabled": True,
        "spawn_on_reset": True,
        "spawn_during_episode": True,
        "vehicles_per_lane": 1,
        "lanes": "ego_and_adjacent",
        "distance_range": [25.0, 70.0],
        "speed_offset_range": [2.0, 6.0],
        "absolute_speed_range": [23.0, 34.0],
        "min_lane_gap": 18.0,
        "spawn_probability": 0.35,
        "cooldown_policy_steps": 3,
        "min_ego_progress": 80.0,
        "max_extra_vehicles": 12,
    }
    traffic_flow_reward_config = {
        "enabled": True,
        "penalty_weight": 0.12,
        "speed_tolerance": 2.0,
        "max_penalty": 0.8,
        "front_ttc_safe": 4.0,
        "rear_ttc_pressure": 5.0,
        "ttc_cap": 10.0,
        "rear_pressure_floor": 0.25,
        "flow_radius": 120.0,
        "lanes": "ego_and_adjacent",
    }
    hyperparameters = {
        "learning_rate": 2.5e-4,
        "buffer_size": 50_000,
        "learning_starts": 2_000,
        "batch_size": 64,
        "gamma": 0.95,
        "target_update_interval": 1_000,
        "train_freq": train_freq,
        "gradient_steps": gradient_steps,
        "exploration_fraction": 0.70,
        "exploration_final_eps": 0.10,
        "progress_every": 1_000,
        "verbose": 0,
    }

    saved_model_eval_episodes = 1000

    def configure_adaptive_experiment(
        *,
        run_name: str,
        adaptive_overrides: dict | None = None,
        rear_flow: dict | None = None,
        traffic_flow_reward: dict | None = None,
        ttc_observation: dict | None = None,
        seed_offset: int = 0,
    ) -> dict:
        adaptive_config = {
            **base_adaptive_longitudinal_config,
            **dict(adaptive_overrides or {}),
        }
        training_env_config = build_env_config(
            profile_name=environment_profile,
            profile_overrides=environment_overrides,
            observation=observation_config,
            action=action_config,
            reward=reward_config,
            speed=speed_config,
            adaptive_longitudinal=adaptive_config,
            rear_flow=rear_flow,
            traffic_flow_reward=traffic_flow_reward,
            ttc_observation=ttc_observation,
        )
        saved_model_eval_env_config = build_env_config(
            profile_name=eval_environment_profile,
            profile_overrides=eval_environment_overrides,
            observation=observation_config,
            action=action_config,
            reward=reward_config,
            speed=speed_config,
            adaptive_longitudinal=adaptive_config,
            rear_flow=rear_flow,
            traffic_flow_reward=traffic_flow_reward,
            ttc_observation=ttc_observation,
        )
        args = build_dqn_args(
            results_dir=RESULTS_DIR,
            run_name=run_name,
            timesteps=timesteps,
            eval_episodes=eval_episodes,
            seed=seed + seed_offset,
            num_envs=n_envs,
            device=DEFAULT_DEVICE,
            hyperparameters=hyperparameters,
        )
        return {
            "run_name": run_name,
            "args": args,
            "training_env_config": training_env_config,
            "saved_model_eval_env_config": saved_model_eval_env_config,
            "saved_model_eval_seed": seed + 10000 + seed_offset,
            "saved_model_eval_name": f"saved_model_eval_{saved_model_eval_episodes}_episodes",
        }

    def run_adaptive_experiment(experiment: dict, label: str) -> dict:
        display(pd.DataFrame({
            "training": pd.Series(experiment["training_env_config"]),
            "saved_eval": pd.Series(experiment["saved_model_eval_env_config"]),
        }))
        summary = train_and_display(
            trainer,
            experiment["args"],
            experiment["training_env_config"],
            label=label,
        )
        saved_eval_summary = evaluate_saved_model(
            trainer,
            summary_path=RESULTS_DIR / experiment["run_name"] / "summary.json",
            env_config=experiment["saved_model_eval_env_config"],
            episodes=saved_model_eval_episodes,
            seed=experiment["saved_model_eval_seed"],
            name=experiment["saved_model_eval_name"],
            label=label,
        )
        return {"summary": summary, "saved_eval_summary": saved_eval_summary}

rear_flow_reward = configure_adaptive_experiment(
run_name="adaptive_rear_flow_reward_20k",
adaptive_overrides={"mode": "delta"},
rear_flow=rear_flow_config,
traffic_flow_reward=traffic_flow_reward_config,
seed_offset=100,
)
rear_flow_reward_results = run_adaptive_experiment(
rear_flow_reward,
label="Adaptive DQN - Rear Flow + Reward",
)


## Experiment 3: Safe-Speed Controller

In [ ]:
if "configure_adaptive_experiment" not in globals():
    import os
    import sys
    from pathlib import Path

    import pandas as pd

    if "build_env_config" not in globals() or "RESULTS_DIR" not in globals():
        def find_project_root() -> Path:
            for candidate in [Path.cwd(), *Path.cwd().parents]:
                if (candidate / "src").exists() and (candidate / ".git").exists():
                    return candidate
            raise RuntimeError("Could not locate the project root.")

        PROJECT_ROOT = find_project_root()
        HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
        helper_dir_str = str(HELPER_DIR)
        if helper_dir_str not in sys.path:
            sys.path.insert(0, helper_dir_str)

        from dqn_notebook_utils import (
            build_dqn_args,
            build_env_config,
            evaluate_saved_model,
            load_dqn_backend,
            train_and_display,
        )

        trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
            backend_module="elurant_dqn",
            notebook_subdir="adaptive_lower_controller",
            results_subdir="adaptive_lower_controller",
        )

    environment_profile = "structured_baseline"
    eval_environment_profile = environment_profile

    environment_overrides = {
        "lanes_count": 3,
        "vehicles_count": 20,
        "duration": 40,
        "ego_spacing": 2.0,
        "vehicles_density": 1.0,
        "simulation_frequency": 15,
        "policy_frequency": 1,
        "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
        "initial_lane_id": None,
        "offroad_terminal": False,
    }
    # Evaluation environment: choose a profile independently from training.
    # Leave overrides empty unless you intentionally want to override that
    # eval profile's defaults.
    eval_environment_overrides = {}

    observation_config = {
        "vehicles_count": 10,
        "features": ["presence", "x", "y", "vx", "vy"],
        "absolute": False,
        "see_behind": True,
    }
    action_config = {
        "type": "DiscreteMetaAction",
    }
    reward_config = {
        "collision_reward": -1.0,
        "right_lane_reward": 0.1,
        "high_speed_reward": 0.4,
        "lane_change_reward": 0.0,
        "normalize_reward": True,
    }
    min_reward_speed = 20.0
    max_reward_speed = 30.0
    speed_config = {
        "reward_speed_range": [min_reward_speed, max_reward_speed],
    }

    timesteps = 20_000
    n_envs = min(4, os.cpu_count() or 1)
    eval_episodes = 5
    seed = 42
    train_freq = 4
    gradient_steps = train_freq * n_envs

    base_adaptive_longitudinal_config = {
        "enabled": True,
        "mode": "delta",
        "ttc_midpoint": 4.0,
        "ttc_temperature": 1.0,
        "ttc_cap": 10.0,
        "min_target_speed": 10.0,
        "max_target_speed": 35.0,
        "faster_max_delta": 1.25,
        "slower_min_delta": 1.25,
        "slower_max_delta": 2.5,
    }
    rear_flow_config = {
        "enabled": True,
        "spawn_on_reset": True,
        "spawn_during_episode": True,
        "vehicles_per_lane": 1,
        "lanes": "ego_and_adjacent",
        "distance_range": [25.0, 70.0],
        "speed_offset_range": [2.0, 6.0],
        "absolute_speed_range": [23.0, 34.0],
        "min_lane_gap": 18.0,
        "spawn_probability": 0.35,
        "cooldown_policy_steps": 3,
        "min_ego_progress": 80.0,
        "max_extra_vehicles": 12,
    }
    traffic_flow_reward_config = {
        "enabled": True,
        "penalty_weight": 0.12,
        "speed_tolerance": 2.0,
        "max_penalty": 0.8,
        "front_ttc_safe": 4.0,
        "rear_ttc_pressure": 5.0,
        "ttc_cap": 10.0,
        "rear_pressure_floor": 0.25,
        "flow_radius": 120.0,
        "lanes": "ego_and_adjacent",
    }
    hyperparameters = {
        "learning_rate": 2.5e-4,
        "buffer_size": 50_000,
        "learning_starts": 2_000,
        "batch_size": 64,
        "gamma": 0.95,
        "target_update_interval": 1_000,
        "train_freq": train_freq,
        "gradient_steps": gradient_steps,
        "exploration_fraction": 0.70,
        "exploration_final_eps": 0.10,
        "progress_every": 1_000,
        "verbose": 0,
    }

    saved_model_eval_episodes = 1000

    def configure_adaptive_experiment(
        *,
        run_name: str,
        adaptive_overrides: dict | None = None,
        rear_flow: dict | None = None,
        traffic_flow_reward: dict | None = None,
        ttc_observation: dict | None = None,
        seed_offset: int = 0,
    ) -> dict:
        adaptive_config = {
            **base_adaptive_longitudinal_config,
            **dict(adaptive_overrides or {}),
        }
        training_env_config = build_env_config(
            profile_name=environment_profile,
            profile_overrides=environment_overrides,
            observation=observation_config,
            action=action_config,
            reward=reward_config,
            speed=speed_config,
            adaptive_longitudinal=adaptive_config,
            rear_flow=rear_flow,
            traffic_flow_reward=traffic_flow_reward,
            ttc_observation=ttc_observation,
        )
        saved_model_eval_env_config = build_env_config(
            profile_name=eval_environment_profile,
            profile_overrides=eval_environment_overrides,
            observation=observation_config,
            action=action_config,
            reward=reward_config,
            speed=speed_config,
            adaptive_longitudinal=adaptive_config,
            rear_flow=rear_flow,
            traffic_flow_reward=traffic_flow_reward,
            ttc_observation=ttc_observation,
        )
        args = build_dqn_args(
            results_dir=RESULTS_DIR,
            run_name=run_name,
            timesteps=timesteps,
            eval_episodes=eval_episodes,
            seed=seed + seed_offset,
            num_envs=n_envs,
            device=DEFAULT_DEVICE,
            hyperparameters=hyperparameters,
        )
        return {
            "run_name": run_name,
            "args": args,
            "training_env_config": training_env_config,
            "saved_model_eval_env_config": saved_model_eval_env_config,
            "saved_model_eval_seed": seed + 10000 + seed_offset,
            "saved_model_eval_name": f"saved_model_eval_{saved_model_eval_episodes}_episodes",
        }

    def run_adaptive_experiment(experiment: dict, label: str) -> dict:
        display(pd.DataFrame({
            "training": pd.Series(experiment["training_env_config"]),
            "saved_eval": pd.Series(experiment["saved_model_eval_env_config"]),
        }))
        summary = train_and_display(
            trainer,
            experiment["args"],
            experiment["training_env_config"],
            label=label,
        )
        saved_eval_summary = evaluate_saved_model(
            trainer,
            summary_path=RESULTS_DIR / experiment["run_name"] / "summary.json",
            env_config=experiment["saved_model_eval_env_config"],
            episodes=saved_model_eval_episodes,
            seed=experiment["saved_model_eval_seed"],
            name=experiment["saved_model_eval_name"],
            label=label,
        )
        return {"summary": summary, "saved_eval_summary": saved_eval_summary}

safe_speed_controller = configure_adaptive_experiment(
run_name="adaptive_safe_speed_controller_20k",
adaptive_overrides={
    "mode": "safe_speed_limiter",
    "min_target_speed": 18.0,
    "cruise_speed": 28.0,
    "action_speed_delta": 3.0,
},
rear_flow=rear_flow_config,
traffic_flow_reward=traffic_flow_reward_config,
seed_offset=200,
)
safe_speed_controller_results = run_adaptive_experiment(
safe_speed_controller,
label="Adaptive DQN - Safe-Speed Controller",
)


## Experiment 4: Wide-Band Delta Controller


In [ ]:
if "configure_adaptive_experiment" not in globals():
    import os
    import sys
    from pathlib import Path

    import pandas as pd

    if "build_env_config" not in globals() or "RESULTS_DIR" not in globals():
        def find_project_root() -> Path:
            for candidate in [Path.cwd(), *Path.cwd().parents]:
                if (candidate / "src").exists() and (candidate / ".git").exists():
                    return candidate
            raise RuntimeError("Could not locate the project root.")

        PROJECT_ROOT = find_project_root()
        HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
        helper_dir_str = str(HELPER_DIR)
        if helper_dir_str not in sys.path:
            sys.path.insert(0, helper_dir_str)

        from dqn_notebook_utils import (
            build_dqn_args,
            build_env_config,
            evaluate_saved_model,
            load_dqn_backend,
            train_and_display,
        )

        trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
            backend_module="elurant_dqn",
            notebook_subdir="adaptive_lower_controller",
            results_subdir="adaptive_lower_controller",
        )

    environment_profile = "structured_baseline"
    eval_environment_profile = environment_profile

    environment_overrides = {
        "lanes_count": 3,
        "vehicles_count": 20,
        "duration": 40,
        "ego_spacing": 2.0,
        "vehicles_density": 1.0,
        "simulation_frequency": 15,
        "policy_frequency": 1,
        "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
        "initial_lane_id": None,
        "offroad_terminal": False,
    }
    eval_environment_overrides = {}

    observation_config = {
        "vehicles_count": 10,
        "features": ["presence", "x", "y", "vx", "vy"],
        "absolute": False,
        "see_behind": True,
    }
    action_config = {
        "type": "DiscreteMetaAction",
    }
    reward_config = {
        "collision_reward": -1.0,
        "right_lane_reward": 0.1,
        "high_speed_reward": 0.4,
        "lane_change_reward": 0.0,
        "normalize_reward": True,
    }
    min_reward_speed = 20.0
    max_reward_speed = 30.0
    speed_config = {
        "reward_speed_range": [min_reward_speed, max_reward_speed],
    }

    timesteps = 20_000
    n_envs = min(4, os.cpu_count() or 1)
    eval_episodes = 5
    seed = 42
    train_freq = 4
    gradient_steps = train_freq * n_envs

    base_adaptive_longitudinal_config = {
        "enabled": True,
        "mode": "delta",
        "ttc_midpoint": 4.0,
        "ttc_temperature": 1.0,
        "ttc_cap": 10.0,
        "min_target_speed": 10.0,
        "max_target_speed": 35.0,
        "faster_max_delta": 1.25,
        "slower_min_delta": 1.25,
        "slower_max_delta": 2.5,
    }
    rear_flow_config = {
        "enabled": True,
        "spawn_on_reset": True,
        "spawn_during_episode": True,
        "vehicles_per_lane": 1,
        "lanes": "ego_and_adjacent",
        "distance_range": [25.0, 70.0],
        "speed_offset_range": [2.0, 6.0],
        "absolute_speed_range": [23.0, 34.0],
        "min_lane_gap": 18.0,
        "spawn_probability": 0.35,
        "cooldown_policy_steps": 3,
        "min_ego_progress": 80.0,
        "max_extra_vehicles": 12,
    }
    traffic_flow_reward_config = {
        "enabled": True,
        "penalty_weight": 0.12,
        "speed_tolerance": 2.0,
        "max_penalty": 0.8,
        "front_ttc_safe": 4.0,
        "rear_ttc_pressure": 5.0,
        "ttc_cap": 10.0,
        "rear_pressure_floor": 0.25,
        "flow_radius": 120.0,
        "lanes": "ego_and_adjacent",
    }
    hyperparameters = {
        "learning_rate": 2.5e-4,
        "buffer_size": 50_000,
        "learning_starts": 2_000,
        "batch_size": 64,
        "gamma": 0.95,
        "target_update_interval": 1_000,
        "train_freq": train_freq,
        "gradient_steps": gradient_steps,
        "exploration_fraction": 0.70,
        "exploration_final_eps": 0.10,
        "progress_every": 1_000,
        "verbose": 0,
    }

    saved_model_eval_episodes = 1000

    def configure_adaptive_experiment(
        *,
        run_name: str,
        adaptive_overrides: dict | None = None,
        rear_flow: dict | None = None,
        traffic_flow_reward: dict | None = None,
        ttc_observation: dict | None = None,
        seed_offset: int = 0,
    ) -> dict:
        adaptive_config = {
            **base_adaptive_longitudinal_config,
            **dict(adaptive_overrides or {}),
        }
        training_env_config = build_env_config(
            profile_name=environment_profile,
            profile_overrides=environment_overrides,
            observation=observation_config,
            action=action_config,
            reward=reward_config,
            speed=speed_config,
            adaptive_longitudinal=adaptive_config,
            rear_flow=rear_flow,
            traffic_flow_reward=traffic_flow_reward,
            ttc_observation=ttc_observation,
        )
        saved_model_eval_env_config = build_env_config(
            profile_name=eval_environment_profile,
            profile_overrides=eval_environment_overrides,
            observation=observation_config,
            action=action_config,
            reward=reward_config,
            speed=speed_config,
            adaptive_longitudinal=adaptive_config,
            rear_flow=rear_flow,
            traffic_flow_reward=traffic_flow_reward,
            ttc_observation=ttc_observation,
        )
        args = build_dqn_args(
            results_dir=RESULTS_DIR,
            run_name=run_name,
            timesteps=timesteps,
            eval_episodes=eval_episodes,
            seed=seed + seed_offset,
            num_envs=n_envs,
            device=DEFAULT_DEVICE,
            hyperparameters=hyperparameters,
        )
        return {
            "run_name": run_name,
            "args": args,
            "training_env_config": training_env_config,
            "saved_model_eval_env_config": saved_model_eval_env_config,
            "saved_model_eval_seed": seed + 10000 + seed_offset,
            "saved_model_eval_name": f"saved_model_eval_{saved_model_eval_episodes}_episodes",
        }

    def run_adaptive_experiment(experiment: dict, label: str) -> dict:
        display(pd.DataFrame({
            "training": pd.Series(experiment["training_env_config"]),
            "saved_eval": pd.Series(experiment["saved_model_eval_env_config"]),
        }))
        summary = train_and_display(
            trainer,
            experiment["args"],
            experiment["training_env_config"],
            label=label,
        )
        saved_eval_summary = evaluate_saved_model(
            trainer,
            summary_path=RESULTS_DIR / experiment["run_name"] / "summary.json",
            env_config=experiment["saved_model_eval_env_config"],
            episodes=saved_model_eval_episodes,
            seed=experiment["saved_model_eval_seed"],
            name=experiment["saved_model_eval_name"],
            label=label,
        )
        return {"summary": summary, "saved_eval_summary": saved_eval_summary}

# Wider-band controller: keep continuous target-speed control, but restore the
# highway-env-sized action bandwidth. FASTER can add up to +10 m/s and SLOWER
# can range from 0 to -10 m/s, with TTC controlling how much of that command is used.
wide_band_adaptive_overrides = {
    "mode": "delta",
    "min_target_speed": 20.0,
    "max_target_speed": 30.0,
    "faster_max_delta": 10.0,
    "slower_min_delta": 0.0,
    "slower_max_delta": 10.0,
    "ttc_temperature": 0.75,
}

wide_band_delta_controller = configure_adaptive_experiment(
    run_name="adaptive_wide_band_delta_20k",
    adaptive_overrides=wide_band_adaptive_overrides,
    rear_flow=rear_flow_config,
    traffic_flow_reward=traffic_flow_reward_config,
    seed_offset=300,
)
wide_band_delta_results = run_adaptive_experiment(
    wide_band_delta_controller,
    label="Adaptive DQN - Wide-Band Delta Controller",
)


## Experiment 5: TTC-Augmented State


In [ ]:
# Refresh helper/backend definitions so this cell works after earlier stale notebook cells.
import importlib

import dqn_notebook_utils as dqn_utils

dqn_utils = importlib.reload(dqn_utils)
build_dqn_args = dqn_utils.build_dqn_args
build_env_config = dqn_utils.build_env_config
evaluate_saved_model = dqn_utils.evaluate_saved_model
train_and_display = dqn_utils.train_and_display
trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = dqn_utils.load_dqn_backend(
    backend_module="elurant_dqn",
    notebook_subdir="adaptive_lower_controller",
    results_subdir="adaptive_lower_controller",
)

_required_adaptive_names = [
    "environment_profile",
    "eval_environment_profile",
    "environment_overrides",
    "eval_environment_overrides",
    "observation_config",
    "action_config",
    "reward_config",
    "speed_config",
    "base_adaptive_longitudinal_config",
    "hyperparameters",
    "timesteps",
    "eval_episodes",
    "saved_model_eval_episodes",
    "n_envs",
    "seed",
]
_missing_adaptive_names = [name for name in _required_adaptive_names if name not in globals()]
if _missing_adaptive_names:
    raise RuntimeError(f"Run the Common Config cell first. Missing: {_missing_adaptive_names}")


def configure_adaptive_experiment(
    *,
    run_name: str,
    adaptive_overrides: dict | None = None,
    rear_flow: dict | None = None,
    traffic_flow_reward: dict | None = None,
    ttc_observation: dict | None = None,
    seed_offset: int = 0,
) -> dict:
    adaptive_config = {
        **base_adaptive_longitudinal_config,
        **dict(adaptive_overrides or {}),
    }
    training_env_config = build_env_config(
        profile_name=environment_profile,
        profile_overrides=environment_overrides,
        observation=observation_config,
        action=action_config,
        reward=reward_config,
        speed=speed_config,
        adaptive_longitudinal=adaptive_config,
        rear_flow=rear_flow,
        traffic_flow_reward=traffic_flow_reward,
        ttc_observation=ttc_observation,
    )
    saved_model_eval_env_config = build_env_config(
        profile_name=eval_environment_profile,
        profile_overrides=eval_environment_overrides,
        observation=observation_config,
        action=action_config,
        reward=reward_config,
        speed=speed_config,
        adaptive_longitudinal=adaptive_config,
        rear_flow=rear_flow,
        traffic_flow_reward=traffic_flow_reward,
        ttc_observation=ttc_observation,
    )
    args = build_dqn_args(
        results_dir=RESULTS_DIR,
        run_name=run_name,
        timesteps=timesteps,
        eval_episodes=eval_episodes,
        seed=seed + seed_offset,
        num_envs=n_envs,
        device=DEFAULT_DEVICE,
        hyperparameters=hyperparameters,
    )
    return {
        "run_name": run_name,
        "args": args,
        "training_env_config": training_env_config,
        "saved_model_eval_env_config": saved_model_eval_env_config,
        "saved_model_eval_seed": seed + 10000 + seed_offset,
        "saved_model_eval_name": f"saved_model_eval_{saved_model_eval_episodes}_episodes",
    }

# Add TTC as an explicit continuous feature in the kinematic observation matrix.
ttc_observation_config = {
    "enabled": True,
    "feature_name": "ttc",
    "ttc_cap": 10.0,
    "lane_y_threshold": 0.35,
    "front_only": True,
    "normalize": True,
}

ttc_observation_adaptive_overrides = globals().get(
    "wide_band_adaptive_overrides",
    {
        "mode": "delta",
        "min_target_speed": 20.0,
        "max_target_speed": 30.0,
        "faster_max_delta": 10.0,
        "slower_min_delta": 0.0,
        "slower_max_delta": 10.0,
        "ttc_temperature": 0.75,
    },
)

ttc_observation_controller = configure_adaptive_experiment(
    run_name="adaptive_ttc_observation_20k",
    adaptive_overrides=ttc_observation_adaptive_overrides,
    rear_flow=rear_flow_config,
    traffic_flow_reward=traffic_flow_reward_config,
    ttc_observation=ttc_observation_config,
    seed_offset=400,
)
ttc_observation_results = run_adaptive_experiment(
    ttc_observation_controller,
    label="Adaptive DQN - Wide-Band + TTC Observation",
)


## Experiment 6: TTC Safety Override Controller


In [ ]:
# Refresh helper/backend definitions so this cell works after earlier stale notebook cells.
import importlib

import dqn_notebook_utils as dqn_utils

dqn_utils = importlib.reload(dqn_utils)
build_dqn_args = dqn_utils.build_dqn_args
build_env_config = dqn_utils.build_env_config
evaluate_saved_model = dqn_utils.evaluate_saved_model
train_and_display = dqn_utils.train_and_display
trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = dqn_utils.load_dqn_backend(
    backend_module="elurant_dqn",
    notebook_subdir="adaptive_lower_controller",
    results_subdir="adaptive_lower_controller",
)

_required_adaptive_names = [
    "environment_profile",
    "eval_environment_profile",
    "environment_overrides",
    "eval_environment_overrides",
    "observation_config",
    "action_config",
    "reward_config",
    "speed_config",
    "base_adaptive_longitudinal_config",
    "hyperparameters",
    "timesteps",
    "eval_episodes",
    "saved_model_eval_episodes",
    "n_envs",
    "seed",
]
_missing_adaptive_names = [name for name in _required_adaptive_names if name not in globals()]
if _missing_adaptive_names:
    raise RuntimeError(f"Run the Common Config cell first. Missing: {_missing_adaptive_names}")


def configure_adaptive_experiment(
    *,
    run_name: str,
    adaptive_overrides: dict | None = None,
    rear_flow: dict | None = None,
    traffic_flow_reward: dict | None = None,
    ttc_observation: dict | None = None,
    seed_offset: int = 0,
) -> dict:
    adaptive_config = {
        **base_adaptive_longitudinal_config,
        **dict(adaptive_overrides or {}),
    }
    training_env_config = build_env_config(
        profile_name=environment_profile,
        profile_overrides=environment_overrides,
        observation=observation_config,
        action=action_config,
        reward=reward_config,
        speed=speed_config,
        adaptive_longitudinal=adaptive_config,
        rear_flow=rear_flow,
        traffic_flow_reward=traffic_flow_reward,
        ttc_observation=ttc_observation,
    )
    saved_model_eval_env_config = build_env_config(
        profile_name=eval_environment_profile,
        profile_overrides=eval_environment_overrides,
        observation=observation_config,
        action=action_config,
        reward=reward_config,
        speed=speed_config,
        adaptive_longitudinal=adaptive_config,
        rear_flow=rear_flow,
        traffic_flow_reward=traffic_flow_reward,
        ttc_observation=ttc_observation,
    )
    args = build_dqn_args(
        results_dir=RESULTS_DIR,
        run_name=run_name,
        timesteps=timesteps,
        eval_episodes=eval_episodes,
        seed=seed + seed_offset,
        num_envs=n_envs,
        device=DEFAULT_DEVICE,
        hyperparameters=hyperparameters,
    )
    return {
        "run_name": run_name,
        "args": args,
        "training_env_config": training_env_config,
        "saved_model_eval_env_config": saved_model_eval_env_config,
        "saved_model_eval_seed": seed + 10000 + seed_offset,
        "saved_model_eval_name": f"saved_model_eval_{saved_model_eval_episodes}_episodes",
    }

# Safety override: the lower controller computes target speed from TTC directly.
# High-level FASTER/SLOWER magnitude is ignored; unsafe FASTER requests below 1s TTC get an extra penalty.
ttc_safety_override_config = {
    "mode": "ttc_safety_override",
    "min_target_speed": 20.0,
    "max_target_speed": 30.0,
    "ttc_midpoint": 4.0,
    "ttc_temperature": 0.75,
    "ttc_cap": 10.0,
    "safety_ttc_threshold": 1.0,
    "unsafe_action_penalty": 1.0,
}

ttc_safety_override_controller = configure_adaptive_experiment(
    run_name="adaptive_ttc_safety_override_20k",
    adaptive_overrides=ttc_safety_override_config,
    rear_flow=rear_flow_config,
    traffic_flow_reward=traffic_flow_reward_config,
    seed_offset=500,
)
ttc_safety_override_results = run_adaptive_experiment(
    ttc_safety_override_controller,
    label="Adaptive DQN - TTC Safety Override Controller",
)


## Congestion Failure Diagnostics

Apply the congestion taxonomy to saved adaptive lower-controller experiments and compare the failure modes across controller formulations.


In [ ]:
import json
from pathlib import Path

from stable_baselines3 import DQN

from congestion_diagnostics import evaluate_congestion_diagnostics, save_congestion_diagnostics

adaptive_diagnostic_episodes = 100
adaptive_diagnostic_seed = seed + 60_000
adaptive_diagnostic_config = {
    "ttc_cap": 10.0,
    "front_ttc_safe": 4.0,
    "front_ttc_critical": 1.5,
    "rear_ttc_safe": 4.0,
    "rear_ttc_critical": 1.5,
    "lane_gap_safe": 12.0,
    "bad_action_margin": 0.35,
    "no_good_action_risk": 0.85,
    "wrong_lane_quality_margin": 0.18,
    "wrong_lane_lookback_steps": 6,
    "final_lookback_steps": 4,
}

adaptive_diagnostic_run_names = [
    "adaptive_rear_flow_env_20k",
    "adaptive_rear_flow_reward_20k",
    "adaptive_safe_speed_controller_20k",
    "adaptive_wide_band_delta_20k",
    "adaptive_ttc_observation_20k",
    "adaptive_ttc_safety_override_20k",
]

adaptive_diagnostic_outputs = []
adaptive_diagnostic_frames = []
for diagnostic_run_name in adaptive_diagnostic_run_names:
    summary_path = RESULTS_DIR / diagnostic_run_name / "summary.json"
    if not summary_path.exists():
        print(f"Skipping missing run: {summary_path}")
        continue

    saved_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    diagnostic_model = DQN.load(saved_summary["model_path"], device=DEFAULT_DEVICE)
    diagnostic_env_config = saved_summary["env_config"]
    output_dir = RESULTS_DIR / diagnostic_run_name / f"congestion_diagnostics_{adaptive_diagnostic_episodes}_episodes"

    diagnostic_df, diagnostic_traces = evaluate_congestion_diagnostics(
        diagnostic_model,
        trainer.make_env,
        env_config=diagnostic_env_config,
        episodes=adaptive_diagnostic_episodes,
        seed=adaptive_diagnostic_seed,
        diagnostic_config=adaptive_diagnostic_config,
    )
    diagnostic_paths = save_congestion_diagnostics(diagnostic_df, diagnostic_traces, output_dir)
    diagnostic_df.insert(0, "run_name", diagnostic_run_name)
    adaptive_diagnostic_frames.append(diagnostic_df)
    adaptive_diagnostic_outputs.append({"run_name": diagnostic_run_name, **diagnostic_paths})

if not adaptive_diagnostic_frames:
    raise RuntimeError("No saved adaptive lower-controller runs were found for congestion diagnostics.")

adaptive_congestion_df = pd.concat(adaptive_diagnostic_frames, ignore_index=True)
label_columns = [
    "collision",
    "agent_chose_badly",
    "no_good_discrete_action",
    "wrong_lane_earlier",
    "unavoidable_rear_pressure",
]
adaptive_label_rates = (
    adaptive_congestion_df.groupby("run_name")[label_columns]
    .mean()
    .mul(100.0)
    .round(2)
    .reset_index()
)
adaptive_collision_breakdown = (
    adaptive_congestion_df.groupby(["run_name", "collision_type"])
    .size()
    .reset_index(name="episodes")
)

print(json.dumps(adaptive_diagnostic_outputs, indent=2))
display(adaptive_label_rates)
display(adaptive_collision_breakdown)
display(adaptive_congestion_df.head(20))
